# 9 - Panel data

A two-level MultiIndex is read as (entity, time) - the standard econometric
layout - and recorded in the frame's metadata.

In [ ]:
%load_ext econenv
import econenv, numpy as np, pandas as pd

In [ ]:
rng = np.random.default_rng(5)
countries = ['DZA', 'MAR', 'TUN', 'EGY', 'JOR']
years = pd.date_range('2000-01-01', periods=20, freq='YS')
index = pd.MultiIndex.from_product([countries, years], names=['country', 'year'])

panel = pd.DataFrame(index=index)
panel['x'] = rng.normal(size=len(index))
effects = pd.Series({c: rng.normal(scale=2) for c in countries})
panel['y'] = (1.0 + 0.6 * panel.x
              + panel.index.get_level_values('country').map(effects)
              + rng.normal(scale=0.5, size=len(index)))
panel.head()

## EconEnv reads the structure automatically

In [ ]:
meta = econenv.schema.describe_frame(panel, name='panel')
print('panel variable:', meta.panel_var)
print('time variable :', meta.time_var)
print('frequency     :', meta.frequency)

## Stata

In [ ]:
econenv.push('stata', 'default', panel)

In [ ]:
%%stata
encode country, gen(cid)
gen y_int = year
xtset cid y_int, yearly
xtreg y x, fe

## R

In [ ]:
%%R -i panel -o fe_table
if (requireNamespace('plm', quietly = TRUE)) {
  fit <- plm::plm(y ~ x, data = panel, index = c('country','year'), model = 'within')
  fe_table <- as.data.frame(summary(fit)$coefficients)
} else {
  fit <- lm(y ~ x + factor(country), data = panel)
  fe_table <- as.data.frame(coef(summary(fit)))[1:2, ]
  message('plm not installed - used LSDV instead')
}

In [ ]:
fe_table

## A note on scope

`econenv.compare_ols` covers OLS in v0.1. Panel FE/RE are in the model
registry marked `planned` - run `%econ models` to see the full matrix. Until
they land, use each engine's own command as above.

In [ ]:
%econ models